# Quantization from scratch
**LLMs From Basics · M5.1 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/m5-1-quantization.ipynb)

In the lesson you quantized nine weights by hand. Here you'll write the quantizers from the lesson in NumPy and measure what they cost:

1. Symmetric quantization (int8, int4)
2. Asymmetric quantization with a zero-point
3. Per-tensor vs per-channel vs group-wise scales
4. What one outlier does
5. Quantize a weight matrix and compare matmul outputs

Each exercise has a `# TODO` to fill in, followed by a check cell with `assert`s. Solutions are at the end. Runs on CPU in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__)

## 1. Symmetric quantization

With $b$ bits and symmetric quantization, the codes are the integers $-q_{\max} \ldots q_{\max}$ with $q_{\max} = 2^{b-1} - 1$ (127 for int8, 7 for int4). The scale is set by the largest magnitude:

$$s = \frac{\max|x|}{q_{\max}}, \qquad q = \mathrm{clip}(\mathrm{round}(x / s),\ -q_{\max},\ q_{\max}), \qquad \hat x = s\,q$$

Write `quantize_symmetric(x, bits)` returning the integer codes `q` (an integer array) and the scale `s`, and `dequantize(q, s, z=0)` returning $s\,(q - z)$.

In [ ]:
def quantize_symmetric(x, bits):
    # TODO: compute qmax, the scale s, and the integer codes q
    raise NotImplementedError

def dequantize(q, s, z=0):
    # TODO: return s * (q - z) as floats
    raise NotImplementedError

In [ ]:
x = rng.normal(size=1000)
for bits in (8, 4):
    q, s = quantize_symmetric(x, bits)
    qmax = 2 ** (bits - 1) - 1
    assert np.issubdtype(q.dtype, np.integer), "codes must be integers"
    assert q.min() >= -qmax and q.max() <= qmax
    assert np.isclose(s, np.abs(x).max() / qmax)
    err = np.abs(x - dequantize(q, s))
    assert err.max() <= s / 2 + 1e-12, "rounding error can't exceed s/2"
    print(f"int{bits}: scale {s:.5f}, max error {err.max():.5f}, RMS error {np.sqrt((err**2).mean()):.5f}")
print("✓ symmetric quantization works")

## 2. Asymmetric quantization with a zero-point

For data that isn't centred on zero, fit the grid to $[x_{\min}, x_{\max}]$ with codes $0 \ldots 2^b - 1$:

$$s = \frac{x_{\max} - x_{\min}}{2^b - 1}, \qquad z = -\mathrm{round}(x_{\min} / s), \qquad q = \mathrm{clip}(\mathrm{round}(x/s) + z,\ 0,\ 2^b - 1)$$

Write `quantize_asymmetric(x, bits)` returning `q, s, z`. The check uses the worked example from the lesson.

In [ ]:
def quantize_asymmetric(x, bits):
    # TODO: compute s, the integer zero-point z, and the codes q
    raise NotImplementedError

In [ ]:
x = np.array([-1.20, -0.35, 0.00, 0.47, 1.80])
q, s, z = quantize_asymmetric(x, 4)
assert np.isclose(s, 0.2) and z == 6, (s, z)
assert q.tolist() == [0, 4, 6, 8, 15], q
assert np.allclose(dequantize(q, s, z), [-1.2, -0.4, 0.0, 0.4, 1.8])
# zero must come back exactly
assert dequantize(q, s, z)[2] == 0.0
# lopsided data: asymmetric uses the grid better than symmetric
y = rng.uniform(0, 1, size=2000)
qa, sa, za = quantize_asymmetric(y, 4)
qs, ss = quantize_symmetric(y, 4)
ea = np.sqrt(np.mean((y - dequantize(qa, sa, za)) ** 2))
es = np.sqrt(np.mean((y - dequantize(qs, ss)) ** 2))
print(f"all-positive data, 4-bit RMS error: asymmetric {ea:.4f} vs symmetric {es:.4f}")
assert ea < es
print("✓ asymmetric quantization works")

## 3. Per-tensor, per-channel and group-wise scales

Write `quant_error(W, bits, mode, group_size=32)` that quantizes the matrix `W` (shape: out × in) **symmetrically**, dequantizes it, and returns the relative RMS error $\sqrt{\sum (W - \hat W)^2 / \sum W^2}$.

- `mode="tensor"`: one scale for the whole matrix
- `mode="channel"`: one scale per row (output channel)
- `mode="group"`: one scale per block of `group_size` consecutive weights in each row (assume the row length is divisible by `group_size`)

Hint: reshape so each scale's numbers form one row, use `keepdims=True` for the max, then reshape back.

In [ ]:
def fake_quant_symmetric(W, bits, mode, group_size=32):
    # TODO: return the dequantized matrix W_hat, same shape as W
    raise NotImplementedError

def quant_error(W, bits, mode, group_size=32):
    W_hat = fake_quant_symmetric(W, bits, mode, group_size)
    return np.sqrt(np.sum((W - W_hat) ** 2) / np.sum(W ** 2))

In [ ]:
# rows with very different sizes, as in real layers
row_scale = np.exp(rng.normal(0, 1.0, size=(256, 1)))
W = rng.normal(size=(256, 512)) * row_scale
errs = {m: quant_error(W, 4, m) for m in ("tensor", "channel", "group")}
for m, e in errs.items():
    print(f"int4 {m:8s}: relative RMS error {e:.4f}")
assert errs["channel"] < errs["tensor"], "per-channel should beat per-tensor here"
assert errs["group"] < errs["channel"], "group-wise (32) should beat per-channel"
assert quant_error(W, 8, "channel") < quant_error(W, 4, "channel") / 10
print("✓ granularity matters")

## 4. One outlier

Take a well-behaved matrix and plant **one** weight that is 50× the standard deviation. Compute the relative RMS error of int8 quantization **on all the other weights** (use a boolean mask that excludes the outlier):

- `err_clean`: per-tensor, no outlier
- `err_outlier`: per-tensor, with the outlier
- `err_outlier_group`: group-wise (32), with the outlier

In [ ]:
W = rng.normal(size=(128, 256))
W_out = W.copy()
W_out[3, 7] = 50.0
mask = np.ones_like(W, dtype=bool)
mask[3, 7] = False

def masked_rel_err(W, W_hat, mask):
    d = (W - W_hat)[mask]
    return np.sqrt(np.sum(d ** 2) / np.sum(W[mask] ** 2))

# TODO: compute the three errors with fake_quant_symmetric(..., bits=8, ...)
err_clean = None
err_outlier = None
err_outlier_group = None

In [ ]:
print(f"int8 per-tensor, clean:        {err_clean:.4f}")
print(f"int8 per-tensor, one outlier:  {err_outlier:.4f}  ({err_outlier / err_clean:.1f}× worse)")
print(f"int8 group-32,   one outlier:  {err_outlier_group:.4f}")
assert err_outlier > 5 * err_clean, "one outlier should make per-tensor int8 much worse"
assert err_outlier_group < err_outlier / 5, "group-wise scales should contain the damage"
print("✓ one outlier can wreck a shared scale")

## 5. Quantize a layer, compare its outputs

What matters isn't the weight error but how much the layer's **output** changes. With activations `X` (tokens × in) and weights `W` (out × in), the layer computes `Y = X @ W.T`.

Compute the relative error $\lVert Y - \hat Y \rVert / \lVert Y \rVert$ (Frobenius norms) for:

- `err_int8`: int8 per-channel weights
- `err_int4`: int4 per-channel weights
- `err_int4_g`: int4 group-wise (32) weights

Activations stay in full precision (this is "weight-only" quantization, W8A16 / W4A16).

In [ ]:
X = rng.normal(size=(64, 512))
W = rng.normal(size=(256, 512)) * np.exp(rng.normal(0, 0.5, size=(256, 1)))
Y = X @ W.T

def out_err(W_hat):
    # TODO: relative Frobenius error of the layer output
    raise NotImplementedError

err_int8 = None    # TODO
err_int4 = None    # TODO
err_int4_g = None  # TODO

In [ ]:
print(f"output error  int8 per-channel: {err_int8:.4%}")
print(f"output error  int4 per-channel: {err_int4:.4%}")
print(f"output error  int4 group-32:    {err_int4_g:.4%}")
assert err_int8 < 0.01
assert err_int4_g < err_int4 < 0.2
assert err_int4 > 5 * err_int8
print("✓ int8 is nearly lossless; int4 needs finer scales (and smarter methods like GPTQ/AWQ)")

## Experiment

Plot the relative weight error against the number of bits for the three granularities. Each extra bit should roughly halve the error (a straight line on a log scale). Then try:

- add an outlier to `W` and re-plot: which curve moves most?
- change `group_size` to 8, 64 or 128. What do you gain and what does it cost in storage? (An fp16 scale per group adds `16 / group_size` bits per weight.)

In [ ]:
W = rng.normal(size=(256, 512)) * np.exp(rng.normal(0, 1.0, size=(256, 1)))
bits_list = [2, 3, 4, 5, 6, 8]
plt.figure(figsize=(6, 4))
for mode in ("tensor", "channel", "group"):
    plt.semilogy(bits_list, [quant_error(W, b, mode) for b in bits_list], "o-", label=mode)
plt.xlabel("bits per weight")
plt.ylabel("relative RMS error")
plt.title("Symmetric round-to-nearest quantization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## Solutions
Only run these if you're stuck. They redefine the functions from the exercises.

### Solution 1

In [ ]:
def quantize_symmetric(x, bits):
    qmax = 2 ** (bits - 1) - 1
    s = np.max(np.abs(x)) / qmax
    q = np.clip(np.round(x / s), -qmax, qmax).astype(np.int32)
    return q, s

def dequantize(q, s, z=0):
    return s * (q.astype(np.float64) - z)

### Solution 2

In [ ]:
def quantize_asymmetric(x, bits):
    levels = 2 ** bits - 1
    s = (np.max(x) - np.min(x)) / levels
    z = int(-np.round(np.min(x) / s))
    q = np.clip(np.round(x / s) + z, 0, levels).astype(np.int32)
    return q, s, z

### Solution 3

In [ ]:
def fake_quant_symmetric(W, bits, mode, group_size=32):
    qmax = 2 ** (bits - 1) - 1
    if mode == "tensor":
        blocks = W.reshape(1, -1)
    elif mode == "channel":
        blocks = W.reshape(W.shape[0], -1)
    elif mode == "group":
        blocks = W.reshape(-1, group_size)
    else:
        raise ValueError(mode)
    s = np.abs(blocks).max(axis=1, keepdims=True) / qmax
    s = np.where(s == 0, 1.0, s)
    W_hat = s * np.clip(np.round(blocks / s), -qmax, qmax)
    return W_hat.reshape(W.shape)

def quant_error(W, bits, mode, group_size=32):
    W_hat = fake_quant_symmetric(W, bits, mode, group_size)
    return np.sqrt(np.sum((W - W_hat) ** 2) / np.sum(W ** 2))

### Solution 4

In [ ]:
W = rng.normal(size=(128, 256))
W_out = W.copy()
W_out[3, 7] = 50.0
mask = np.ones_like(W, dtype=bool)
mask[3, 7] = False

def masked_rel_err(W, W_hat, mask):
    d = (W - W_hat)[mask]
    return np.sqrt(np.sum(d ** 2) / np.sum(W[mask] ** 2))

err_clean = masked_rel_err(W, fake_quant_symmetric(W, 8, "tensor"), mask)
err_outlier = masked_rel_err(W_out, fake_quant_symmetric(W_out, 8, "tensor"), mask)
err_outlier_group = masked_rel_err(W_out, fake_quant_symmetric(W_out, 8, "group", 32), mask)

### Solution 5

In [ ]:
X = rng.normal(size=(64, 512))
W = rng.normal(size=(256, 512)) * np.exp(rng.normal(0, 0.5, size=(256, 1)))
Y = X @ W.T

def out_err(W_hat):
    Y_hat = X @ W_hat.T
    return np.linalg.norm(Y - Y_hat) / np.linalg.norm(Y)

err_int8 = out_err(fake_quant_symmetric(W, 8, "channel"))
err_int4 = out_err(fake_quant_symmetric(W, 4, "channel"))
err_int4_g = out_err(fake_quant_symmetric(W, 4, "group", 32))